# Task 01 (Hosted online at (http://elizachatbot.somee.com/))

In [ ]:
import re
import random


# Reflections used to guess the pronoun
REFLECTIONS = {
    "i am"    : "you are",
    "i was"   : "you were",
    "i've"    : "you have",
    "i'll"    : "you will",
    "i'd"     : "you would",
    "i'm"     : "you are",
    "i"       : "you",
    "my"      : "your",
    "me"      : "you",
    "myself"  : "yourself",
    "you are" : "I am",
    "you were": "I was",
    "you've"  : "I have",
    "you'll"  : "I will",
    "your"    : "my",
    "yours"   : "mine",
    "you"     : "me",
    "am"      : "are",
    "was"     : "were",
}


def reflect(text: str) -> str:
    """
    Swap pronouns so the bot can echo the user's words naturally.
    Checks two-word phrases before single words to avoid partial matches.

    Example: "I feel like I am failing" → "you feel like you are failing"
    """
    tokens = text.lower().split()
    result = []
    i = 0
    while i < len(tokens):
        two = ' '.join(tokens[i:i+2])
        if two in REFLECTIONS:
            result.append(REFLECTIONS[two])
            i += 2
        elif tokens[i] in REFLECTIONS:
            result.append(REFLECTIONS[tokens[i]])
            i += 1
        else:
            result.append(tokens[i])
            i += 1
    return ' '.join(result)

# Regex patterns to check the sentences and guess their response

PATTERNS = [

    #  Greetings
    (r'\b(hello|hi|hey)\b|good (morning|evening|afternoon)',
     [
         "Hello! I'm here to listen and support you. How are you feeling today?",
         "Hi there. This is a safe space to share. What's on your mind?",
         "Hey! I'm glad you reached out. How are you doing today?",
     ]),

    (r'(?:my name is|i am called|call me|this is|i am|i\'m)\s+([a-zA-Z]+)',
     [
         "Hello {0}! It's good to meet you. How are you feeling today?",
         "Nice to meet you, {0}! What would you like to talk about?",
         "Welcome, {0}. I'm here to listen. How are things going for you?",
         "Great to have you here, {0}. How are you doing today?",
     ]),

    #  CRISIS — must come BEFORE generic "I" patterns
    (r'(suicide|kill myself|end my life|want to die|don.t want to live)',
     [
         "I'm very concerned about what you just shared. Please reach out for help right now.\n"
         "  Umang Helpline (Pakistan): 0317-4288665\n"
         "  International: www.findahelpline.com\n"
         "You are not alone, and people care about you.",

         "What you're feeling sounds very serious and your safety matters most. "
         "Please contact a crisis line immediately.\n"
         "  Umang (Pakistan): 0317-4288665\n"
         "Your life has value.",
     ]),

    (r'(self.harm|hurt myself|hurting myself|cutting myself)',
     [
         "I'm really concerned about your safety right now. Please speak to a trusted person "
         "or contact a helpline immediately.\n"
         "  Umang (Pakistan): 0317-4288665\n"
         "You deserve care and support.",
     ]),

    #  Panic / physical anxiety symptoms
    (r'(panic attack|can.t breathe|heart is racing|chest is tight|chest tight)',
     [
         "It sounds like you might be experiencing a panic attack. "
         "Let's slow down together — breathe in for 4 counts, hold for 4, out for 4. "
         "You are safe. What is happening right now?",
         "That sounds very frightening. Try to focus on slow, steady breathing. "
         "I'm here with you. Can you tell me what triggered this?",
     ]),

    #  Anxiety
    (r'(anxious|anxiety|panic|panicking|nervous|worried|worrying)',
     [
         "I'm sorry to hear you're struggling with anxiety. "
         "Can you tell me more about what's triggering these feelings?",
         "Anxiety can feel overwhelming. What thoughts are running through your mind right now?",
         "It takes courage to acknowledge anxiety. When did these feelings start?",
         "You're not alone — many people experience this. What seems to be setting it off?",
     ]),

    #  Depression / Sadness
    (r'i feel (sad|depressed|hopeless|empty|numb|miserable|worthless)',
     [
         "I'm sorry you're feeling {0}. Would you like to share what's been going on?",
         "Feeling {0} can be really heavy. How long have you been feeling this way?",
         "Thank you for trusting me with this. What do you think is contributing to feeling {0}?",
         "You don't have to carry this alone. What has your day been like?",
     ]),

    (r'(nothing matters|pointless|no point|what.s the point|life is meaningless)',
     [
         "It sounds like you're feeling very hopeless right now. "
         "Can you tell me what's been happening that's brought you to this point?",
         "Those feelings of emptiness are very painful. Has something specific happened recently?",
         "I hear you. When did things start feeling this way for you?",
     ]),

    #  Stress / Overwhelm
    (r'(stressed|overwhelmed|too much|can.t cope|can.t handle|burned out)',
     [
         "It sounds like you're carrying a lot right now. What feels most overwhelming?",
         "Stress can build up so quickly. What are the main things weighing on you today?",
         "I hear you. Let's take this one step at a time. "
         "What is the single biggest source of stress for you right now?",
         "You're doing the right thing by talking about it. "
         "What would help you feel even slightly better right now?",
     ]),

    #  Loneliness
    (r'(lonely|alone|isolated|no one understands|no one cares|no friends)',
     [
         "Feeling lonely is one of the hardest experiences. "
         "What does your support network look like right now?",
         "I'm here and I'm listening. Can you tell me more about feeling alone?",
         "You reached out, which means part of you is seeking connection — "
         "that's a meaningful step. What's been making you feel isolated?",
     ]),

    #  Anger
    (r'(i feel angry|i am angry|i.m angry|i.m frustrated|furious|rage|anger)',
     [
         "Anger often signals that something important to us has been hurt or crossed. "
         "What happened?",
         "It's okay to feel angry. Can you walk me through what led to this?",
         "I understand. What do you think is underneath the anger?",
     ]),

    #  Sleep problems
    (r'(can.t sleep|insomnia|not sleeping|sleep problems|awake at night|trouble sleeping)',
     [
         "Sleep problems can make everything feel so much harder. "
         "How long have you been struggling with this?",
         "Lack of sleep affects our mood, thinking and energy a lot. "
         "What do you think is keeping you awake?",
         "That sounds exhausting. Are your thoughts racing when you try to sleep?",
     ]),

    #  Family / Relationships
    (r'my (family|parents|mother|father|partner|husband|wife|friend|relationship)',
     [
         "Relationships can be really complicated. What's been happening with your {0}?",
         "Tell me more about the situation with your {0}.",
         "How does the situation with your {0} make you feel?",
         "What do you wish your {0} understood about what you're going through?",
     ]),

    #  Work / Study
    (r'(university|college|studies|exam|assignment|work|job|career|grades)',
     [
         "Academic and work pressure can be very intense. What's been happening with {0}?",
         "Tell me more about the pressure you're feeling around {0}.",
         "How is the situation with {0} affecting your daily life and wellbeing?",
     ]),

    #  Seeking professional help
    (r'(therapist|therapy|counselor|counselling|psychiatrist|psychologist)',
     [
         "Reaching out for professional support is one of the bravest steps you can take. "
         "Have you been able to see a {0} before?",
         "That's a really positive thought. What's making you consider seeing a {0}?",
         "A {0} can offer support that goes much deeper than what I can provide. "
         "I really encourage you to reach out to one.",
     ]),

    #  What can the bot do
    (r'(what can you do|how can you help|can you help)',
     [
         "I can listen without judgment, reflect your feelings back to you, "
         "and help you think through what's on your mind. "
         "I'm not a replacement for professional therapy, but I'm a good first step. "
         "What would you like to talk about?",
         "I'm here to offer a safe, non-judgmental space. Tell me what's been going on.",
     ]),

    #  Who is the bot
    (r'(who are you|what are you|are you a robot|are you human|are you real)',
     [
         "I am an ELIZA-type mental health support chatbot inspired by Joseph Weizenbaum's "
         "original ELIZA (1966). I'm not a human or a licensed therapist, "
         "but I'm here to listen and help you reflect.",
         "I'm a chatbot designed to provide a supportive space to talk through your feelings. "
         "For serious concerns, please reach out to a qualified mental health professional.",
     ]),

    #  Coping strategies
    (r'(what should i do|how do i cope|how can i feel better|what helps|give me advice)',
     [
         "That's a great question. What has helped you get through difficult moments before?",
         "Coping looks different for everyone. What small things bring you even a little comfort?",
         "Let's think about this together. Are there activities or people that usually "
         "help you feel more grounded?",
     ]),

    #  Feeling better / Positive
    (r'i feel (good|better|happy|grateful|relieved|calm|hopeful)',
     [
         "I'm really glad to hear that! What's been going well?",
         "That's wonderful to hear. What do you think has helped you feel better?",
         "It's great to hear some positivity. What's bringing you this sense of {0}?",
     ]),

    #  Thank you
    (r'\b(thank you|thanks|thank u|thx)\b',
     [
         "You're very welcome. I'm glad I could be here for you. Is there anything else on your mind?",
         "Of course. That's what I'm here for. How are you feeling right now?",
     ]),

    #  Goodbye
    (r'\b(bye|goodbye|quit|exit|see you|take care)\b',
     [
         "Take care of yourself. Remember, it's always okay to ask for help. Goodbye!",
         "Goodbye! I hope our conversation was helpful. "
         "Please reach out to a professional if things feel too heavy.",
         "Take good care. You matter, and so does your mental health. Goodbye!",
     ]),

    #  I don't know
    (r'(i don.t know|i.m not sure|i can.t explain|hard to explain|i don.t understand)',
     [
         "That's perfectly okay. Take your time. "
         "Sometimes our feelings are difficult to put into words.",
         "It's alright not to have all the answers. "
         "What does it feel like, even if you can't name it exactly?",
         "Just being here and trying to talk about it is already meaningful. "
         "What's the closest you can get to describing how things feel?",
     ]),

    #  I feel ... (generic catch-all for emotions)
    (r'i feel (.*)',
     [
         "Why do you feel {0}?",
         "How long have you been feeling {0}?",
         "What do you think is making you feel {0}?",
         "Does feeling {0} affect your daily life?",
         "What would help you feel less {0}?",
         "Tell me more about feeling {0}.",
     ]),

    #  I ... (generic catch-all)
    (r'^i (.*)',
     [
         "Why do you {0}?",
         "Can you tell me more about that?",
         "How does it make you feel when you {0}?",
         "How long have you felt this way?",
         "What do you mean when you say you {0}?",
     ]),

    #  Final fallback — always matches
    (r'(.*)',
     [
         "I see. Can you tell me more about that?",
         "How does that make you feel?",
         "Let's explore that a little more. Can you say more?",
         "I'm listening. Please go on.",
         "That's important. What else is on your mind?",
         "How long have you been thinking about this?",
         "What do you think that means for you?",
     ]),
]


# Function to respond
# 1. Match user input
# 2. iterate through patterns
# 3. Pick a random response for a capture group
# 4. Return the string
def respond(user_input: str) -> str:


    text = user_input.lower().strip()

    for pattern, responses in PATTERNS:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            response = random.choice(responses)
            # Replace {0} with the reflected capture group (if present in both)
            if match.lastindex and '{0}' in response:
                captured  = match.group(1).strip('.,!?')
                reflected = reflect(captured)
                response  = response.replace('{0}', reflected)
            return response

    return "I'm here to listen. Can you tell me more?"


# Menu for the user
def chat():
    print("  Type 'bye' to end the session.")
    print("=" * 65)
    print()
    print("Chatbot : Hello! I'm here to listen. How are you feeling today?")
    print()

    while True:
        try:
            user_input = input("You     : ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nChatbot : Take care. Goodbye!")
            break

        if not user_input:
            print("Chatbot : I'm here whenever you're ready to share.")
            print()
            continue

        response = respond(user_input)
        print(f"Chatbot : {response}")
        print()

        if re.search(r'\b(bye|goodbye|quit|exit)\b', user_input, re.IGNORECASE):
            break

# Demo
def demo():
    """Simulate a complete conversation covering all major patterns."""
    conversation = [
        "Hello, my name is Maryam.",
        "I have been feeling very anxious lately.",
        "I can't sleep at night.",
        "I feel sad most of the time.",
        "I feel like nothing matters anymore.",
        "I don't know what to do.",
        "My family doesn't understand me.",
        "Sometimes I feel very lonely.",
        "What can you do to help me?",
        "Who are you?",
        "Should I see a therapist?",
        "Thank you for listening.",
        "I feel a little better now.",
        "Goodbye.",
    ]

    print("=" * 65)
    print("   ELIZA Mental Health Support Chatbot — DEMO")
    print("   Instructor: Dr. Ijaz Hussain | Topic: Mental Health")
    print("=" * 65)
    print()
    print("Chatbot : Hello! I'm here to listen. How are you feeling today?")
    print()

    for user_msg in conversation:
        print(f"User    : {user_msg}")
        response = respond(user_msg)
        print(f"Chatbot : {response}")
        print()

if __name__ == '__main__':
        chat()

   ELIZA Mental Health Support Chatbot
   Inspired by ELIZA — Joseph Weizenbaum (1966)

  This is a safe, non-judgmental space.
  IMPORTANT: I am not a real therapist.
  In crisis? Call Umang (Pakistan): 0317-4288665
  Type 'bye' to end the session.

Chatbot : Hello! I'm here to listen. How are you feeling today?

You     : hi
Chatbot : Hello! I'm here to listen and support you. How are you feeling today?

You     : i am feeling sad 
Chatbot : Why do you are feeling sad?

You     : people do not understand me 
Chatbot : I see. Can you tell me more about that?

You     : i got less marks in my exams
Chatbot : How is the situation with exam affecting your daily life and wellbeing?

You     : i do not want to talk
Chatbot : How does it make you feel when you do not want to talk?

You     : bye
Chatbot : Take care of yourself. Remember, it's always okay to ask for help. Goodbye!



# Task 02

#For txt file

In [ ]:
"""
Urdu Sentence Segmentation & Tokenization
==========================================

Pipeline:
  1. Load corpus (txt or xlsx)
  2. Clean text (Remove diacritics, remove the URLs, English numbers etc)
  3. Tokenize into words
  4. Segment into sentences (based on punctuation, starting and ending words)
  5. Evaluate (Precision, Recall, F1)
  6. Save results to CSV
"""

import re
import os
import pandas as pd
from typing import List, Optional
from collections import Counter


# Making dictionary for starting, ending and punctuation words

URDU_TERMINATORS = {'۔', '؟', '!'}

SENTENCE_END_WORDS = {
    'ہے', 'ہیں', 'تھا', 'تھی', 'تھے',
    'گا',  'گی',  'گے',
    'ہوا', 'ہوئی', 'ہوئے',
    'کیا', 'کی',  'کئے',
    'دیا', 'دی',  'دیے',
    'آیا', 'آئی', 'آئے',
    'گیا', 'گئی', 'گئے',
    'لیا', 'لی',  'لیے',
    'رہا', 'رہی', 'رہے',
    'سکا', 'سکی', 'سکے',
    'چکا', 'چکی', 'چکے',
    'ملا', 'ملی', 'ملے',
    'بنا', 'بنی', 'بنے',
    'جائے', 'آئے',
    'کرے', 'کریں',
    'ہو',  'ہوں',
    'کریگا', 'کریگی',
    'ہوگا', 'ہوگی', 'ہوگے',
    'کرتا', 'کرتی', 'کرتے',
    'چاہیے',
    'سکتا', 'سکتی', 'سکتے',
    'پایا', 'پائی', 'پائے',
    'ہوجاتا', 'ہوجاتی', 'ہوجاتے',
    'ہوجائے', 'کرلیا', 'کرلی',
    'نہیں', 'ہاں', 'ضرور', 'بھی',
    'ھے',  'ھیں',
    'ھا',  'ھی',   'ھے',
    'ھوا', 'ھوئی', 'ھوئے',
    'ھوگا','ھوگی', 'ھوگے',
    'ھو',  'ھوں',
    'ھوتا','ھوتی', 'ھوتے',
    'رھا', 'رھی',  'رھے',
    'کرتھا','کرتھی',
    'دیا', 'لیا',
}

SENTENCE_START_WORDS = {
    'اس',   'یہ',    'وہ',    'جب',
    'لیکن', 'مگر',   'پھر',   'اب',
    'تو',   'لہذا',  'اگر',   'کیونکہ',
    'چونکہ','چونکه', 'اگرچہ', 'حالانکہ',
    'جبکہ', 'تاکہ',  'جیسا',  'جیسے',
    'اس لئے', 'اس لیے',
    'جو',   'جس',   'کاش',
    'پہلے', 'آج',   'کل',
    'پس',   'لہٰذا',
}


# Function to normalize text
def normalize_whitespace(text: str) -> str:
    text = text.replace('\u200c', ' ').replace('\u200d', '').replace('\xa0', ' ')
    text = text.replace('\r\n', ' ').replace('\r', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r' ([۔؟!،؛])', r'\1', text)
    text = re.sub(r'([۔؟!])([^\s])', r'\1 \2', text)
    return text.strip()


def remove_diacritics(text: str) -> str:
    """
    Remove Arabic/Urdu diacritical marks (harakat) so that
    dictionary lookups match regardless of vowel marks.
    """
    return re.sub(
        r'[\u0610-\u061A\u064B-\u065F\u0670'
        r'\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]',
        '', text
    )


def clean_urdu_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.replace('\r\n', ' ').replace('\r', ' ')
    text = re.sub(r'http\S+|www\.\S+', '', text)           # URLs
    text = re.sub(r'#\w+', '', re.sub(r'@\w+', '', text))                        # #hashtags
    text = re.sub(r'[a-zA-Z0-9]', '', text)                # English & digits
    text = re.sub(r'[^\u0600-\u06FF\s۔؟!،؛]', ' ', text)  # non-Urdu chars
    return normalize_whitespace(text)


# ─────────────────────────────────────────────────
# 3.  CORPUS LOADERS
# ─────────────────────────────────────────────────

def load_txt_corpus(filepath: str) -> pd.DataFrame:
    """
    Load a plain text Urdu corpus (.txt).
    Expects one sentence / entry per line.

    Returns DataFrame with columns: text, label
    """
    print(f"\n[LOADER] Reading: {os.path.basename(filepath)}")
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    lines = [ln.strip().replace('\r', '') for ln in lines]
    lines = [ln for ln in lines if ln]
    print(f"  Lines read : {len(lines)}")
    return pd.DataFrame({'text': lines, 'label': None})


def load_xlsx_corpus(filepath: str,
                     text_col: str = None,
                     label_col: str = None,
                     sheet_name=0) -> pd.DataFrame:
    print(f"\n[LOADER] Reading: {os.path.basename(filepath)}")
    df_raw = pd.read_excel(filepath, sheet_name=sheet_name, engine='openpyxl')
    print(f"  Shape   : {df_raw.shape}")
    print(f"  Columns : {list(df_raw.columns)}")

    text_candidates  = ['Text', 'text', 'tweet', 'Tweet', 'sentence',
                        'Sentence', 'urdu', 'Urdu', 'content', 'Content']
    label_candidates = ['Label', 'label', 'class', 'Class',
                        'category', 'Category', 'hate', 'Hate']

    if text_col is None:
        text_col = next((c for c in text_candidates if c in df_raw.columns), None)
        if text_col is None:
            str_cols = df_raw.select_dtypes(include='object').columns.tolist()
            text_col = str_cols[0] if str_cols else None
        if text_col is None:
            raise ValueError("Cannot find text column. Pass text_col='your_column_name'.")

    if label_col is None:
        label_col = next((c for c in label_candidates if c in df_raw.columns), None)

    print(f"  Text column  : '{text_col}'")
    print(f"  Label column : '{label_col}'" if label_col else "  Label column : not found")

    return pd.DataFrame({
        'text':  df_raw[text_col].astype(str),
        'label': df_raw[label_col].astype(str) if label_col else None,
    })


def load_corpus(filepath: str, **kwargs) -> pd.DataFrame:
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.txt':
        return load_txt_corpus(filepath)
    elif ext in ('.xlsx', '.xls'):
        return load_xlsx_corpus(filepath, **kwargs)
    else:
        raise ValueError(f"Unsupported file type '{ext}'. Use .txt or .xlsx")


# ─────────────────────────────────────────────────
# 4.  TOKENIZER
# ─────────────────────────────────────────────────

def urdu_tokenize(text: str) -> List[str]:
    if not isinstance(text, str) or not text.strip():
        return []
    text = normalize_whitespace(text)
    text = re.sub(r'([۔؟!،؛])', r' \1 ', text)
    return [t.strip() for t in text.split() if t.strip()]


# ─────────────────────────────────────────────────
# 5.  SENTENCE SEGMENTER
# ─────────────────────────────────────────────────

class UrduSentenceSegmenter:


    def __init__(self, end_words=None, start_words=None, min_len=3):
        self.end_words   = end_words   or SENTENCE_END_WORDS
        self.start_words = start_words or SENTENCE_START_WORDS
        self.min_len     = min_len   # minimum words required before a split

    # Strategy 1 (split on space after a punctuation)
    def _split_on_terminators(self, text: str) -> List[str]:
        parts = re.split(r'(?<=[۔؟!])\s+', text)
        return [p.strip() for p in parts if p.strip()]

    # Strategy 2 (Split on ending and starting word)
    def _endword_heuristic(self, chunk: str) -> List[str]:
        chunk = chunk.strip()
        has_terminator = chunk and chunk[-1] in URDU_TERMINATORS
        core = chunk[:-1].strip() if has_terminator else chunk

        words = core.split()
        results, current = [], []

        for i, word in enumerate(words):
            clean = remove_diacritics(word).strip('،؛""\'')
            current.append(word)

            if clean in self.end_words and i + 1 < len(words):
                nxt = remove_diacritics(words[i + 1]).strip('،؛""\'')
                if nxt in self.start_words and len(current) >= self.min_len:
                    results.append(' '.join(current) + '۔')
                    current = []

        # Remaining words form the last sentence
        if current:
            tail = ' '.join(current)
            # Restore original terminator on the very last piece
            tail = tail + chunk[-1] if has_terminator else tail
            results.append(tail)

        return results if results else [chunk]

    # Function to insert terminator at end if there is none
    def _ensure_terminator(self, s: str) -> str:
        s = s.strip()
        return s + '۔' if s and s[-1] not in URDU_TERMINATORS else s

    # Main pipeline
    def segment(self, text: str) -> List[str]:
        """
        Full segmentation pipeline for one text entry:
          normalize → terminator-split → heuristic → ensure terminator
        """
        text = normalize_whitespace(text)

        # Strategy 1: split on explicit terminators
        chunks = self._split_on_terminators(text)

        # Strategy 2: apply heuristic on every chunk
        sentences = []
        for chunk in chunks:
            sentences.extend(self._endword_heuristic(chunk))

        # Ensure every sentence ends with a terminator, then normalize spacing
        sentences = [self._ensure_terminator(s) for s in sentences if s.strip()]
        return [normalize_whitespace(s) for s in sentences]


# ─────────────────────────────────────────────────
# 6.  EVALUATION
# ─────────────────────────────────────────────────

def evaluate_segmentation(predicted: List[str], ground_truth: List[str]) -> dict:
    """
    Evaluate segmentation using End-of-Sentence boundary positions.

    Method:
      Flatten both sentence lists into cumulative word-index positions
      where each sentence ends. Compare the two position sets.

      TP = boundary in BOTH predicted and gold
      FP = boundary predicted but NOT in gold  (over-segmentation)
      FN = gold boundary NOT predicted          (under-segmentation)
    """
    def eos_positions(sents):
        pos, total = set(), 0
        for s in sents:
            total += len(s.split())
            pos.add(total - 1)
        return pos

    pred_eos = eos_positions(predicted)
    gold_eos = eos_positions(ground_truth)

    tp = len(pred_eos & gold_eos)
    fp = len(pred_eos - gold_eos)
    fn = len(gold_eos - pred_eos)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return dict(
        precision=round(precision, 4), recall=round(recall, 4),
        f1_score=round(f1, 4), TP=tp, FP=fp, FN=fn,
        predicted=len(predicted), gold=len(ground_truth)
    )


def print_report(metrics: dict):
    print("\n" + "="*52)
    print("       SEGMENTATION EVALUATION REPORT")
    print("="*52)
    print(f"  Predicted sentences : {metrics['predicted']}")
    print(f"  Gold sentences      : {metrics['gold']}")
    print(f"  True Positives (TP) : {metrics['TP']}")
    print(f"  False Positives(FP) : {metrics['FP']}")
    print(f"  False Negatives(FN) : {metrics['FN']}")
    print(f"  {'─'*48}")
    print(f"  Precision           : {metrics['precision']:.4f}")
    print(f"  Recall              : {metrics['recall']:.4f}")
    print(f"  F1 Score            : {metrics['f1_score']:.4f}")
    print("="*52 + "\n")


# ─────────────────────────────────────────────────
# 7.  CORPUS STATISTICS
# ─────────────────────────────────────────────────

def print_corpus_stats(df: pd.DataFrame, source: str):
    total_tokens    = df['tokens'].apply(len).sum()
    total_sentences = df['sentences'].apply(len).sum()
    vocab_size      = df['_vocab_size'].iloc[0]
    multi           = (df['sentence_count'] > 1).sum()

    print("\n" + "="*52)
    print(f"  CORPUS STATISTICS — {os.path.basename(source)}")
    print("="*52)
    print(f"  Total entries        : {len(df)}")
    print(f"  Total tokens         : {total_tokens:,}")
    print(f"  Total sentences      : {total_sentences:,}")
    print(f"  Vocabulary size      : {vocab_size:,}")
    print(f"  Avg tokens / entry   : {df['token_count'].mean():.2f}")
    print(f"  Avg sentences/ entry : {df['sentence_count'].mean():.2f}")
    print(f"  Entries with >1 sent : {multi} ({100*multi/len(df):.1f}%)")

    if 'label' in df.columns and df['label'].notna().any():
        print("\n  Label distribution:")
        for label, count in df['label'].value_counts().items():
            print(f"    {str(label):<20} : {count:>5}  ({100*count/len(df):.1f}%)")
    print("="*52 + "\n")


# ─────────────────────────────────────────────────
# 8.  MAIN PIPELINE
# ─────────────────────────────────────────────────

def run_pipeline(filepath: str):
    """
    Run the full NLP pipeline on a .txt or .xlsx corpus file.
    Results are saved as a CSV in the same folder as the input file.
    """
    print("\n" + "="*60)
    print("   URDU SENTENCE SEGMENTATION & TOKENIZATION PIPELINE")
    print(f"   File : {os.path.basename(filepath)}")
    print("="*60)

    # ── Step 1: Load ─────────────────────────────────────────────
    print("\n[STEP 1] Loading corpus ...")
    df = load_corpus(filepath)
    print(f"  Loaded {len(df)} entries.")

    # ── Step 2: Clean ────────────────────────────────────────────
    print("\n[STEP 2] Cleaning text ...")
    df['clean'] = df['text'].apply(clean_urdu_text)
    before = len(df)
    df = df[df['clean'].str.strip().astype(bool)].reset_index(drop=True)
    print(f"  Entries after cleaning : {len(df)} / {before}")

    # ── Step 3: Tokenize ─────────────────────────────────────────
    print("\n[STEP 3] Tokenizing ...")
    df['tokens']      = df['clean'].apply(urdu_tokenize)
    df['token_count'] = df['tokens'].apply(len)

    vocab = Counter(
        tok for toks in df['tokens'] for tok in toks
        if tok not in {'۔', '؟', '!', '،', '؛'}
    )
    df['_vocab_size'] = len(vocab)

    print(f"  Total tokens     : {df['token_count'].sum():,}")
    print(f"  Vocabulary       : {len(vocab):,} unique words")
    print(f"  Avg tokens/entry : {df['token_count'].mean():.2f}")
    print(f"  Top 10 words     : {[w for w, _ in vocab.most_common(10)]}")

    # ── Step 4: Segment ──────────────────────────────────────────
    print("\n[STEP 4] Segmenting sentences ...")
    segmenter = UrduSentenceSegmenter()
    df['sentences']      = df['clean'].apply(segmenter.segment)
    df['sentence_count'] = df['sentences'].apply(len)

    print(f"  Total sentences      : {df['sentence_count'].sum():,}")
    print(f"  Avg sentences/entry  : {df['sentence_count'].mean():.2f}")
    multi = (df['sentence_count'] > 1).sum()
    print(f"  Entries with >1 sent : {multi} ({100*multi/len(df):.1f}%)")

    # ── Step 5: Evaluate ─────────────────────────────────────────
    print("\n[STEP 5] Evaluating segmentation ...")
    terminated = df[df['clean'].str.contains('۔', na=False)]
    sample     = terminated.head(200)
    print(f"  Using {len(sample)} terminated entries as gold standard ...")

    gold_flat, pred_flat = [], []
    for _, row in sample.iterrows():
        gold_sents = [s.strip() for s in
                      re.split(r'(?<=[۔؟!])\s+', row['clean']) if s.strip()]
        raw        = row['clean'].replace('۔', '').replace('؟', '')
        pred_sents = segmenter.segment(raw)
        gold_flat.extend(gold_sents)
        pred_flat.extend(pred_sents)

    metrics = evaluate_segmentation(pred_flat, gold_flat)
    print_report(metrics)

    # ── Corpus stats ─────────────────────────────────────────────
    print_corpus_stats(df, filepath)

    # ── Step 6: Sample output ────────────────────────────────────
    print("[STEP 6] Sample entries (5 random):")
    print("="*60)
    for i, row in df.sample(min(5, len(df)), random_state=42).iterrows():
        label_str = f"  |  Label: {row['label']}" if row.get('label') else ""
        print(f"\n  Entry #{i+1}{label_str}")
        print(f"  Original  : {str(row['text'])[:90]}")
        print(f"  Cleaned   : {str(row['clean'])[:90]}")
        print(f"  Tokens    : {row['tokens'][:8]}{'...' if row['token_count'] > 8 else ''}")
        print(f"  Sentences :")
        for j, s in enumerate(row['sentences'], 1):
            print(f"    [{j}] {s}")

    # ── Step 7: Save CSV in the same folder as the input file ────
    folder      = os.path.dirname(os.path.abspath(filepath))
    base_name   = os.path.splitext(os.path.basename(filepath))[0]
    output_path = os.path.join(folder, f"{base_name}_tokenized.csv")

    output = df[['text', 'label', 'clean', 'token_count', 'sentence_count']].copy()
    output['tokens']    = df['tokens'].apply(lambda t: ' | '.join(t))
    output['sentences'] = df['sentences'].apply(lambda s: ' || '.join(s))
    output.to_csv(output_path, index=False, encoding='utf-8-sig')

    # print(f"\n[STEP 7] Results saved → {output_path}")
    # print("\n" + "="*60)

    print("  PIPELINE COMPLETE")
    print("="*60 + "\n")

    return df

# ─────────────────────────────────────────────────
# 10. ENTRY POINT
# ─────────────────────────────────────────────────

if __name__ == '__main__':

    FOLDER = os.getcwd()

    # FOLDER    = os.path.dirname(os.path.abspath(__file__))
    TXT_PATH  = os.path.join(FOLDER, 'UrduCorpus.txt')
    XLSX_PATH = os.path.join(FOLDER, 'HS_Urdu_Dataset.xlsx')

    # Running the pipeline for either a txt file or a xlsx file found in the folder 
    if os.path.exists(TXT_PATH):
        df = run_pipeline(TXT_PATH)
    elif os.path.exists(XLSX_PATH):
        df = run_pipeline(XLSX_PATH)
    else:
        print("\n[!] No corpus file found in the script folder.")
        print(f"    Looked for : {TXT_PATH}")
        print(f"    Looked for : {XLSX_PATH}")
        print("    Place either file in the same folder as this script and run again.")


   URDU SENTENCE SEGMENTATION & TOKENIZATION PIPELINE
   File : UrduCorpus.txt

[STEP 1] Loading corpus ...

[LOADER] Reading: UrduCorpus.txt
  Lines read : 4767
  Loaded 4767 entries.

[STEP 2] Cleaning text ...
  Entries after cleaning : 4760 / 4767

[STEP 3] Tokenizing ...
  Total tokens     : 25,618
  Vocabulary       : 2,591 unique words
  Avg tokens/entry : 5.38
  Top 10 words     : ['ہے', 'میں', 'زین', 'نے', 'ہوں', 'نہیں', 'کیا', 'یہ', 'وہ', 'آپ']

[STEP 4] Segmenting sentences ...
  Total sentences      : 4,779
  Avg sentences/entry  : 1.00
  Entries with >1 sent : 19 (0.4%)

[STEP 5] Evaluating segmentation ...
  Using 200 terminated entries as gold standard ...

       SEGMENTATION EVALUATION REPORT
  Predicted sentences : 203
  Gold sentences      : 200
  True Positives (TP) : 200
  False Positives(FP) : 3
  False Negatives(FN) : 0
  ────────────────────────────────────────────────
  Precision           : 0.9852
  Recall              : 1.0000
  F1 Score            : 0.9926
